# Поиск эталонной проводки: зачисление эквайринга Сбера

Источник: платёжное поручение от коллеги.

- **Плательщик:** Северо-Западный банк ПАО Сбербанк (ИНН `7707083893`)
- **Получатель (клиент РСХБ):** ООО «КАЛА Я МАРЬЯПОЯТ» (ИНН `1017000071`), сч. `40702810035530000002`
- **Ожидаемое назначение:** `ЗАЧИСЛЕНИЕ СРЕДСТВ ПО ОПЕРАЦИЯМ ЭКВАЙРИНГА. МЕРЧАНТ №251000008495. ...`
- **Дата / сумма:** ~21–22.06.2026, `68932.64`

Работаем **только** с `ods.scd1_z_main_docum` (без джойнов к другим таблицам).

In [ ]:
import re
import time

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_rows', 200)

print('Imports loaded')

In [ ]:
# === Эталон с фото ===
table_name = 'ods.scd1_z_main_docum'

client_account = '40702810035530000002'
client_inn = '1017000071'
payer_account = '30233810825000117000'
payer_inn_sber = '7707083893'
merchant_id = '251000008495'
amount_expected = 68932.64

nazn_sample = (
    'ЗАЧИСЛЕНИЕ СРЕДСТВ ПО ОПЕРАЦИЯМ ЭКВАЙРИНГА. '
    'МЕРЧАНТ №251000008495. '
    'КОМИССИЯ 1 372.36 (В Т.Ч. НДС 247.47). '
    'ВОЗВРАТ ПОКУПКИ 0.00/0.00.'
)

# Период вокруг проводки (проведена 22.06.2026)
date_from = '2026-06-20'
date_to_exclusive = '2026-06-23'
# Ищем по одному дню — меньше скан ORC, меньше риск Memory limit exceeded
search_dates = ['2026-06-21', '2026-06-22']

# Impala
impala_db = 'sandbox_ai'
impala_queue = 'ai'
impala_user_name = 'Shestopalov-VYur'
impala_keytab_path = '/home/jovyan/test_requests/tech.keytab'
impala_use_credentials = True
impala_update_keytab = True
# 8g на полном скане main_docum не хватает (ошибка ORC allocate)
mem_limit = '32g'
preview_limit = 50
stop_on_first_hit = True  # не гонять тяжёлые like/rlike, если счёт уже нашёл

out_hit_path = './sber_acq_credit_gold_hit.csv'
out_similar_nazn_path = './sber_acq_credit_similar_nazn.csv'

print(f'table={table_name}')
print(f'period=[{date_from}, {date_to_exclusive}), search_dates={search_dates}')
print(f'mem_limit={mem_limit}, stop_on_first_hit={stop_on_first_hit}')
print(f'client_account={client_account}, merchant_id={merchant_id}')
print(f'expected amount={amount_expected}')
print('sample nazn:')
print(nazn_sample)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': impala_queue},
    kerberos={
        'keytab_path': impala_keytab_path,
        'use_credentials': impala_use_credentials,
        'update_keytab': impala_update_keytab,
    },
    user_params={'user_name': impala_user_name},
)
imp._init_connection()
print('Impala connection initialized')

## 1) Схема таблицы — какие колонки есть для счёта / ИНН / суммы / назначения

In [ ]:
def pick_first_existing(columns, candidates):
    colset = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in colset:
            return colset[cand.lower()]
    return None


def find_columns_by_substrings(columns, substrings):
    found = []
    for col in columns:
        low = str(col).lower()
        if any(s in low for s in substrings):
            found.append(col)
    return found


def sql_digits_only(expr):
    return f"regexp_replace(trim(cast({expr} as string)), '[^0-9]', '')"


t0 = time.perf_counter()
with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    schema_raw_df = imp.fetch(f'DESCRIBE {table_name}')
print(f'DESCRIBE done in {round(time.perf_counter() - t0, 2)}s')

name_col = pick_first_existing(schema_raw_df.columns, ['name', 'col_name', 'column_name', 'field'])
if name_col is None:
    name_col = schema_raw_df.columns[0]

table_columns = [
    str(x).strip()
    for x in schema_raw_df[name_col].tolist()
    if str(x).strip() and not str(x).startswith('#')
]

field_map = {
    'c_nazn': pick_first_existing(table_columns, ['c_nazn', 'nazn']),
    'c_date_prov': pick_first_existing(table_columns, ['c_date_prov', 'date_prov', 'c_date']),
    # на кластере сумма = c_sum (см. рабочий SELECT с фото)
    'amount': pick_first_existing(table_columns, ['c_sum', 'c_summa', 'summa', 'amount', 'c_sum_nat']),
    'acc_dt': pick_first_existing(table_columns, ['c_acc_dt', 'acc_dt', 'c_acc_a']),
    'acc_kt': pick_first_existing(table_columns, ['c_acc_kt', 'acc_kt', 'c_acc_b']),
    'inn_primary': pick_first_existing(
        table_columns,
        ['c_kl_kt_2_inn', 'c_kl_dt_2_inn', 'c_inn', 'inn', 'c_inn_dt', 'c_inn_kt'],
    ),
}

account_cols = find_columns_by_substrings(table_columns, ['acc_dt', 'acc_kt', 'c_acc'])
inn_cols = find_columns_by_substrings(table_columns, ['inn'])
amount_cols = find_columns_by_substrings(table_columns, ['summa', 'c_sum', 'amount'])

field_map_df = pd.DataFrame([{'role': k, 'column': v} for k, v in field_map.items()])
display(field_map_df)
print(f'account_cols: {account_cols[:30]}')
print(f'inn_cols: {inn_cols[:30]}')
print(f'amount_cols: {amount_cols[:30]}')
print(f'total columns: {len(table_columns)}')

nazn_col = field_map['c_nazn']
date_col = field_map['c_date_prov']
amount_col = field_map['amount'] or (amount_cols[0] if amount_cols else None)
acc_dt_col = field_map['acc_dt']
acc_kt_col = field_map['acc_kt']

if nazn_col is None:
    raise RuntimeError('c_nazn not found in schema')
if date_col is None:
    print('WARNING: date column not found — date filter will be skipped')

## 2) Поиск: сумма + мерчант

Счёт раньше не нашёлся (`gold_hit_rows=0`). Ищем иначе:

1. `c_sum = 68932.64` + `c_nazn like '%251000008495%'` (мерчант)
2. `c_sum = 68932.64` + `c_nazn like '%эквайр%'`
3. только сумма `68932.64` за день

Без лишних `cast` в SELECT/WHERE. По дням `2026-06-21`, `2026-06-22`.

In [ ]:
# Поиск без лишних cast: колонки как есть.
# Сумма с платёжки: 68932.64, мерчант: 251000008495

if not amount_col:
    raise RuntimeError('Колонка суммы не найдена (ожидали c_sum)')

select_parts = [nazn_col, date_col, amount_col, acc_dt_col, acc_kt_col]
select_parts += [c for c in ['c_kl_dt_2_inn', 'c_kl_kt_2_inn'] if c in set(inn_cols or [])]
select_parts = [c for c in select_parts if c]
select_sql = ',\n    '.join(select_parts)


from datetime import datetime, timedelta

def next_day(day_str):
    return (datetime.strptime(day_str, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')


def run_lookup(label, where_extra, day, limit=50):
    # Диапазон суток без cast по колонке: c_date_prov >= 'D' and < 'D+1'
    date_filter = (
        f"{date_col} >= '{day}' and {date_col} < '{next_day(day)}'"
        if date_col else '1=1'
    )
    sql = f"""
    select
        {select_sql}
    from {table_name}
    where {date_filter}
      and ({where_extra})
    limit {limit}
    """
    print('=' * 80)
    print(f'{label} | day={day}')
    print(sql)
    t0 = time.perf_counter()
    with imp:
        imp.execute(f'set MEM_LIMIT={mem_limit}')
        df = imp.fetch(sql)
    print(f'rows={len(df):,}, elapsed={round(time.perf_counter() - t0, 2)}s')
    if df is not None and not df.empty:
        display(df.head(preview_limit))
    else:
        print('No rows')
    return df if df is not None else pd.DataFrame()


hit_frames = []

# 1) сумма + номер мерчанта в назначении
for day in search_dates:
    where_amt_merch = (
        f"{amount_col} = {amount_expected} "
        f"and {nazn_col} like '%{merchant_id}%'"
    )
    df = run_lookup('1) sum + merchant_id', where_amt_merch, day=day, limit=50)
    if not df.empty:
        df = df.copy()
        df['lookup_label'] = f'sum+merchant|{day}'
        hit_frames.append(df)
        if stop_on_first_hit:
            break

# 2) сумма + эквайр (если мерчант не попал — вдруг другой формат номера)
if (not hit_frames) or (not stop_on_first_hit):
    for day in search_dates:
        where_amt_ekv = (
            f"{amount_col} = {amount_expected} "
            f"and lower({nazn_col}) like '%эквайр%'"
        )
        df = run_lookup('2) sum + эквайр', where_amt_ekv, day=day, limit=50)
        if not df.empty:
            df = df.copy()
            df['lookup_label'] = f'sum+ekv|{day}'
            hit_frames.append(df)
            if stop_on_first_hit:
                break

# 3) только сумма за день (проверить, есть ли вообще такая сумма)
if (not hit_frames) or (not stop_on_first_hit):
    for day in search_dates:
        df = run_lookup('3) sum only', f'{amount_col} = {amount_expected}', day=day, limit=50)
        if not df.empty:
            df = df.copy()
            df['lookup_label'] = f'sum_only|{day}'
            hit_frames.append(df)
            if stop_on_first_hit:
                break

if hit_frames:
    gold_hit_df = pd.concat(hit_frames, ignore_index=True)
    # дедуп по доступным колонкам
    dedup_cols = [c for c in [nazn_col, date_col, amount_col, acc_dt_col, acc_kt_col] if c and c in gold_hit_df.columns]
    if dedup_cols:
        gold_hit_df = gold_hit_df.drop_duplicates(subset=dedup_cols, keep='first')
else:
    gold_hit_df = pd.DataFrame()

print('=' * 80)
print(f'Total unique hit rows: {len(gold_hit_df):,}')
if not gold_hit_df.empty:
    display(gold_hit_df)
    nazn_series = gold_hit_df[nazn_col] if nazn_col in gold_hit_df.columns else gold_hit_df.iloc[:, 0]
    print('\n--- c_nazn values ---')
    for i, v in enumerate(nazn_series.fillna('').astype(str).unique().tolist(), 1):
        print(f'{i}. {v}')
    gold_hit_df.to_csv(out_hit_path, index=False)
    print(f'Saved: {out_hit_path}')
else:
    print('По сумме/мерчанту не найдено. Проверьте: день проводки, что c_sum именно 68932.64, загрузку в ODS.')

## 3) Похожие назначения: `зачисление … эквайринг` за тот же период

Массовый список уникальных `c_nazn` того же семейства (только `main_docum`).

In [ ]:
# Тяжёлый запрос: гоняем по одному дню и с like (без double rlike на всём периоде)
similar_frames = []
for day in search_dates:
    sql_similar = f"""
    select
        coalesce(cast({nazn_col} as string), '') as c_nazn,
        count(*) as cnt
    from {table_name}
    where cast({date_col} as date) = date '{day}'
      and lower(coalesce(cast({nazn_col} as string), '')) like '%зачислен%'
      and lower(coalesce(cast({nazn_col} as string), '')) like '%эквайр%'
    group by coalesce(cast({nazn_col} as string), '')
    order by cnt desc
    limit 200
    """
    print('=' * 80)
    print(f'similar nazn | day={day}')
    print(sql_similar)
    t0 = time.perf_counter()
    try:
        with imp:
            imp.execute(f'set MEM_LIMIT={mem_limit}')
            df = imp.fetch(sql_similar)
        print(f'rows={len(df):,}, elapsed={round(time.perf_counter() - t0, 2)}s')
        if df is not None and not df.empty:
            df = df.copy()
            df['day'] = day
            similar_frames.append(df)
    except Exception as e:
        print(f'SKIP day={day} due to error: {e}')

if similar_frames:
    similar_nazn_df = (
        pd.concat(similar_frames, ignore_index=True)
        .groupby('c_nazn', as_index=False, sort=False)['cnt']
        .sum()
        .sort_values('cnt', ascending=False)
    )
else:
    similar_nazn_df = pd.DataFrame(columns=['c_nazn', 'cnt'])

display(similar_nazn_df.head(preview_limit))

if not similar_nazn_df.empty:
    similar_nazn_df = similar_nazn_df.copy()
    similar_nazn_df['has_merchant_id'] = (
        similar_nazn_df['c_nazn'].fillna('').astype(str).str.contains(merchant_id, regex=False)
    )
    similar_nazn_df['has_merchant_word'] = (
        similar_nazn_df['c_nazn'].fillna('').astype(str).str.lower().str.contains('мерчант', regex=False)
    )
    display(similar_nazn_df.loc[similar_nazn_df['has_merchant_id']])

similar_nazn_df.to_csv(out_similar_nazn_path, index=False)
print(f'Saved: {out_similar_nazn_path}')

## 4) Итог

In [ ]:
summary = pd.DataFrame([
    {'item': 'table', 'value': table_name},
    {'item': 'period', 'value': f'[{date_from}, {date_to_exclusive})'},
    {'item': 'nazn_col', 'value': nazn_col},
    {'item': 'date_col', 'value': date_col},
    {'item': 'amount_col', 'value': amount_col},
    {'item': 'acc_dt', 'value': acc_dt_col},
    {'item': 'acc_kt', 'value': acc_kt_col},
    {'item': 'gold_hit_rows', 'value': len(gold_hit_df) if 'gold_hit_df' in globals() else 0},
    {'item': 'similar_nazn_rows', 'value': len(similar_nazn_df) if 'similar_nazn_df' in globals() else 0},
    {'item': 'hit_csv', 'value': out_hit_path},
    {'item': 'similar_csv', 'value': out_similar_nazn_path},
])
display(summary)

if 'gold_hit_df' in globals() and not gold_hit_df.empty:
    print('OK: эталонная проводка (или близкие) найдены. Смотрите c_nazn выше.')
else:
    print('NOT FOUND: расширьте период или проверьте, что проводка загружена в ODS.')